# 1.5 Rademacher complexity

How well a class can fit pure random noise: a data-dependent measure of richness that often beats VC dimension.

Rademacher complexity averages the best correlation with random signs on the actual sample. A richer class earns a larger noise-fitting score and therefore a larger generalization penalty.

Save a copy to Drive to edit.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(15)


def sign_vectors(m):
    return np.asarray(list(itertools.product([-1, 1], repeat=m)), dtype=float)


def rademacher(F_outputs):
    F_outputs = np.asarray(F_outputs, dtype=float)
    m = F_outputs.shape[1]
    sigmas = sign_vectors(m)
    values = []
    winners = []
    for sigma in sigmas:
        scores = F_outputs @ sigma / m
        winner = int(np.argmax(scores))
        values.append(float(scores[winner]))
        winners.append(winner)
    return float(np.mean(values)), sigmas, np.asarray(values), np.asarray(winners)


def all_patterns(m):
    return np.asarray(list(itertools.product([-1, 1], repeat=m)), dtype=float)


def sampled_rademacher(F_outputs, draws=5000):
    F_outputs = np.asarray(F_outputs, dtype=float)
    m = F_outputs.shape[1]
    values = []
    for _ in range(draws):
        sigma = rng.choice([-1, 1], size=m)
        values.append(float(np.max(F_outputs @ sigma / m)))
    return float(np.mean(values))


def class_from_capacity(m, k):
    patterns = all_patterns(m)
    return patterns[:k]

## The concept, built once (D1)

Empirical Rademacher complexity is

$$\hat{\mathfrak{R}}_S(F)=\mathbb{E}_{\sigma}\left[\sup_{f\in F}rac{1}{m}\sum_{i=1}^{m}\sigma_i f(x_i)ight].$$

The lesson's two-function class is $F=\{[1,1,1],[1,-1,1]\}$ on $m=3$ points.

In [ ]:
lesson_F = np.array([
    [1, 1, 1],
    [1, -1, 1],
], dtype=float)
lesson_rad, lesson_sigmas, lesson_values, lesson_winners = rademacher(lesson_F)
print("Rademacher complexity:", lesson_rad)
print("per-sign maxima:", lesson_values)

The plan requires the exact enumeration value $0.3333$ over $2^3=8$ sign vectors. A single fixed function should score exactly $0$, and the all-sign-pattern class should score $1$.

In [ ]:
assert np.isclose(round(lesson_rad, 4), 0.3333)
single_rad, _, _, _ = rademacher(np.array([[1, 1, 1]], dtype=float))
full_rad, _, _, _ = rademacher(all_patterns(3))
assert np.isclose(single_rad, 0.0)
assert np.isclose(full_rad, 1.0)
print("single fixed function:", single_rad)
print("all patterns:", full_rad)

## The dataset ladder

D1 has one fixed function, D2 is the lesson two-function class, D3 adds four functions, D4 uses all sign patterns for $m=3$ and $m=4$, and D5 approximates a larger class after exact enumeration becomes expensive.

In [ ]:
rungs = []
rungs.append({"name": "D1 one fixed function", "F": np.array([[1, 1, 1]], dtype=float), "approx": False})
rungs.append({"name": "D2 lesson two functions", "F": lesson_F, "approx": False})
rungs.append({"name": "D3 four-function class", "F": class_from_capacity(3, 4), "approx": False})
rungs.append({"name": "D4 all patterns m=4", "F": all_patterns(4), "approx": False})
large_F = rng.choice([-1, 1], size=(128, 12))
rungs.append({"name": "D5 sampled large class", "F": large_F, "approx": True})

for rung in rungs:
    F = rung["F"]
    print(rung["name"])
    print("  functions=", F.shape[0])
    print("  points=", F.shape[1])
    print("  first rows=", F[:3])

In [ ]:
summary = []
for rung in rungs:
    F = rung["F"]
    if rung["approx"]:
        value = sampled_rademacher(F, draws=5000)
    else:
        value, _, _, _ = rademacher(F)
    summary.append((rung["name"], F.shape[0], F.shape[1], value, rung["approx"]))
print("rung | functions | points | Rhat | approximate")
for name, functions, points, value, approximate in summary:
    print(f"{name}: {functions} | {points} | {value:.4f} | {approximate}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for ax, rung in zip(axes[0], rungs):
    F = rung["F"]
    if rung["approx"]:
        sigma = rng.choice([-1, 1], size=F.shape[1])
        scores = F @ sigma / F.shape[1]
        winner = int(np.argmax(scores))
    else:
        _, sigmas, _, winners = rademacher(F)
        sigma = sigmas[-1]
        winner = int(winners[-1])
    ax.step(np.arange(len(sigma)), sigma, where="mid", label="noise")
    ax.step(np.arange(F.shape[1]), F[winner], where="mid", label="best f")
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(rung["name"])
    ax.grid(True, alpha=0.3)
axes[0, 0].legend()
capacities = [item[1] for item in summary]
values = [item[3] for item in summary]
axes[1, 0].plot(capacities, values, marker="o")
axes[1, 0].set_xscale("log", base=2)
axes[1, 0].set_xlabel("number of functions")
axes[1, 0].set_ylabel("empirical Rademacher complexity")
axes[1, 0].set_title("Payoff curve: complexity vs richness")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: underestimating the supremum. Looking at a few functions or signs gives a lower number, but the bound needs the maximum over the whole class; use exhaustive enumeration on tiny rungs and label approximations later.

In [ ]:
d3 = rungs[2]["F"]
wrong_subset = d3[:2]
wrong_value, _, _, _ = rademacher(wrong_subset)
fixed_value, _, _, _ = rademacher(d3)
print(f"wrong subset-only value={wrong_value:.4f}")
print(f"fixed exhaustive value={fixed_value:.4f}")
print(f"underestimate={fixed_value - wrong_value:.4f}")

## Evaluate it + Practice

- Metric: empirical Rademacher complexity.
- No-skill baseline: use one fixed function regardless of class richness.
- Cheap sanity check: for m=3 exact enumeration has 8 sign vectors.
- Ablation: replace the supremum with the first function and watch the score collapse.
- Failure signals: approximations unlabeled, subset-only maxima, or values outside [0, 1].

### Practice prompts

- Add a third function to the lesson class and recompute exactly.

- Compute the bound contribution 2 Rhat for Rhat=1/3.

- Increase m in the full sign-pattern class and predict Rhat.